In [ ]:
from datetime import datetime

import matplotlib.pyplot as mpl
import pandas as pd
import seaborn as sns
import numpy as np

now = datetime.now().strftime("%d/%m/%Y_%H:%M:%S")

print(f"Last notebook execution: {now}")

## **Read Data**
___

In [ ]:
import pandas as pd
from pathlib import Path


root_dir = Path("../outputs")
dfs_list = []

# Define the target filename
target_file = "final_test_results_detailed.csv"

for file_path in root_dir.rglob(target_file):
    try:
        # Get all parts of the path relative to root
        parts = file_path.relative_to(root_dir).parts
        
        # Expected structure:
        # parts[0] = model_name
        # parts[1] = timestamp (e.g., 2026-08-24_10-30-00)
        # parts[2] = eval folder (e.g., eval_seed_1_stratified_split_True_calibration_False)
        
        if len(parts) < 3:
            print(f"⚠️  Skipping: Path too short - {file_path}")
            continue

        model_name = parts[0]
        # timestamp = parts[1] # We don't strictly need this unless you want to save it
        eval_folder_name = parts[2]
        
        # --- Robust Flag Extraction ---
        # We look for the specific keys and grab the value immediately following them
        def extract_flag(folder_name, key):
            # Split by the key followed by underscore
            if f"{key}_" not in folder_name:
                return None # Key not found
            
            # Get the part after the key
            after_key = folder_name.split(f"{key}_")[1]
            # The value is the first segment before the next underscore (or end of string)
            value = after_key.split("_")[0]
            
            if value.lower() == "true":
                return True
            elif value.lower() == "false":
                return False
            else:
                return None # Found key but value wasn't True/False

        is_stratified = extract_flag(eval_folder_name, "stratified_split")
        is_calibrated = extract_flag(eval_folder_name, "calibration")
        
        # Validate extraction
        if is_stratified is None or is_calibrated is None:
            print(f"⚠️  Skipping: Could not parse flags from '{eval_folder_name}' in {file_path}")
            continue

        # Extract Seed from the folder name as well
        # Format: eval_seed_<seed>_...
        seed_val = None
        if "seed_" in eval_folder_name:
            seed_part = eval_folder_name.split("seed_")[1]
            seed_val = seed_part.split("_")[0]
        else:
            print(f"⚠️  Warning: Could not find seed in '{eval_folder_name}'")
            seed_val = "unknown"

        # Read CSV
        df = pd.read_csv(file_path, sep=";")
        
        # Add Metadata
        df["model"] = model_name
        df["seed"] = seed_val
        df["stratified_split"] = is_stratified
        df["calibrated"] = is_calibrated
        
        dfs_list.append(df)

    except Exception as e:
        print(f"❌ Error processing {file_path}: {e}")

if dfs_list:
    combined_df = pd.concat(dfs_list, ignore_index=True)
    print(f"✅ Successfully combined {len(dfs_list)} files.")
    print(f"📊 Final Shape: {combined_df.shape}")
    print("\nSample of metadata columns:")
    print(combined_df[["model", "seed", "stratified_split", "calibrated"]].head())
else:
    print("❌ No files were successfully loaded.")
    
combined_df.sample(5)

In [ ]:
combined_df.columns

In [ ]:
# Rename 'linear_svc' to 'linear_svm' in the model column
combined_df['model'] = combined_df['model'].replace('linear_svc', 'linear_svm')

# Verify the change
print("Unique models after rename:")
print(combined_df['model'].unique())

In [ ]:
# Counts rows per group directly (no temporary column needed)
instance_counts = combined_df.groupby(
    ['model', 'seed', 'stratified_split', 'calibrated'], 
    as_index=False
).size()

# Rename the resulting column from 'size' to 'num_instances'
instance_counts.rename(columns={'size': 'num_instances'}, inplace=True)

print(instance_counts)

## **Metrics Calculation**
___

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, 
    roc_auc_score, matthews_corrcoef, balanced_accuracy_score, roc_curve
)
from scipy.optimize import brentq
from scipy.interpolate import interp1d
import warnings

# Suppress specific runtime warnings for MCC (handled by try/except logic)
warnings.filterwarnings("ignore", message="invalid value encountered")

def calculate_eer_brentq(y_true, y_scores):
    """Calculates EER using brentq with robust bounds checking."""
    fpr, tpr, thresholds = roc_curve(y_true, y_scores)
    fnr = 1 - tpr
    
    # Check if a solution is possible (curves must cross)
    if len(fpr) < 2 or (np.all(fnr > fpr) or np.all(fnr < fpr)):
        return np.nan
        
    try:
        # Interpolation function for TPR
        tpr_interp = interp1d(fpr, tpr, bounds_error=False, fill_value=(tpr[0], tpr[-1]))
        # Find root where FPR = FNR => FPR = 1 - TPR => FPR + TPR - 1 = 0
        eer = brentq(lambda x: 1. - x - tpr_interp(x), 0., 1., maxiter=100)
        return eer
    except (ValueError, RuntimeError):
        return np.nan

def calculate_metrics(group):
    y_true = group['true_label'].values
    y_pred = group['predicted_label'].values
    y_scores = group['pbb_score'].values
    
    # Validate binary labels (must be 0 and 1)
    unique_labels = np.unique(np.concatenate([y_true, y_pred]))
    if not np.all(np.isin(unique_labels, [0, 1])) or len(unique_labels) > 2:
        # Fallback or skip if labels are not strictly 0/1
        # You might want to map them here if they are boolean or strings
        pass 

    # 1. Accuracy
    acc = accuracy_score(y_true, y_pred)
    
    # 2. Precision
    prec = precision_score(y_true, y_pred, zero_division=0)
    
    # 3. Recall (Sensitivity)
    rec = recall_score(y_true, y_pred, zero_division=0)
    
    # 4. F1 Score
    f1 = f1_score(y_true, y_pred, zero_division=0)
    
    # 5. ROC AUC
    try:
        # Check if both classes are present for AUC
        if len(np.unique(y_true)) < 2:
            roc = np.nan
        else:
            roc = roc_auc_score(y_true, y_scores)
    except ValueError:
        roc = np.nan
        
    # 6. Specificity (Recall of negative class 0)
    try:
        if 0 not in y_true:
            spec = np.nan
        else:
            spec = recall_score(y_true, y_pred, pos_label=0, zero_division=0)
    except ValueError:
        spec = np.nan
    
    # 7. Sensitivity (Same as Recall)
    sens = rec
    
    # 8. Balanced Accuracy
    bal_acc = balanced_accuracy_score(y_true, y_pred)
    
    # 9. MCC
    try:
        mcc = matthews_corrcoef(y_true, y_pred)
    except (ValueError, ZeroDivisionError):
        mcc = np.nan
        
    # 10. EER
    eer = calculate_eer_brentq(y_true, y_scores)
    
    return pd.Series({
        'accuracy': acc,
        'precision': prec,
        'recall': rec,
        'f1_score': f1,
        'roc_auc': roc,
        'specificity': spec,
        'sensitivity': sens,
        'balanced_accuracy': bal_acc,
        'mcc': mcc,
        'eer': eer
    })

# --- Execution ---
# Use sort=False to preserve order and potentially speed up slightly
metrics_df = combined_df.groupby(
    ['model', 'seed', 'stratified_split', 'calibrated'], 
    sort=False
).apply(calculate_metrics).reset_index()

print(f"Metrics calculated for {len(metrics_df)} groups.")

# --- Splitting the DataFrames ---

# 1. Stratified=True AND Calibrated=True
df_strat_calib = metrics_df[
    (metrics_df['stratified_split'] == True) & 
    (metrics_df['calibrated'] == True)
].reset_index(drop=True)

# 2. Stratified=False AND Calibrated=False
df_no_strat_no_calib = metrics_df[
    (metrics_df['stratified_split'] == False) & 
    (metrics_df['calibrated'] == False)
].reset_index(drop=True)

# Verification
print(f"\n✅ Subset (Strat=True, Calib=True): {df_strat_calib.shape}")
print(f"✅ Subset (Strat=False, Calib=False): {df_no_strat_no_calib.shape}")

In [ ]:
df_strat_calib

In [ ]:
df_no_strat_no_calib

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, 
    roc_auc_score, matthews_corrcoef, balanced_accuracy_score, 
    roc_curve, brier_score_loss
)
from scipy.optimize import brentq
from scipy.interpolate import interp1d
import warnings

warnings.filterwarnings("ignore", message="invalid value encountered")

def calculate_ece(y_true, y_prob, n_bins=10):
    """Calculates Expected Calibration Error (ECE)."""
    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    total_samples = len(y_true)
    
    if total_samples == 0:
        return np.nan

    for i in range(n_bins):
        in_bin = (y_prob > bin_boundaries[i]) & (y_prob <= bin_boundaries[i+1])
        # Handle edge case for first bin inclusive of 0
        if i == 0:
            in_bin = (y_prob >= bin_boundaries[i]) & (y_prob <= bin_boundaries[i+1])
            
        prop_in_bin = in_bin.mean()
        
        if prop_in_bin > 0:
            avg_confidence = y_prob[in_bin].mean()
            avg_accuracy = y_true[in_bin].mean()
            ece += np.abs(avg_accuracy - avg_confidence) * prop_in_bin
            
    return ece

def calculate_metrics(group):
    y_true = group['true_label'].values.astype(int)
    y_pred = group['predicted_label'].values.astype(int)
    y_scores = group['pbb_score'].values.astype(float)
    
    # Ensure binary labels are strictly 0/1 for sklearn metrics
    # If your labels are strings '0'/'1', astype(int) handles it. 
    # If they are boolean, astype(int) handles it.

    # 1-9. Existing Metrics
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    
    # ROC AUC
    if len(np.unique(y_true)) < 2:
        roc = np.nan
    else:
        roc = roc_auc_score(y_true, y_scores)
        
    # Specificity
    if 0 not in y_true:
        spec = np.nan
    else:
        spec = recall_score(y_true, y_pred, pos_label=0, zero_division=0)
        
    bal_acc = balanced_accuracy_score(y_true, y_pred)
    
    try:
        mcc = matthews_corrcoef(y_true, y_pred)
    except (ValueError, ZeroDivisionError):
        mcc = np.nan
        
    # EER
    def get_eer(y_t, y_s):
        fpr, tpr, thresholds = roc_curve(y_t, y_s)
        fnr = 1 - tpr
        if len(fpr) < 2 or (np.all(fnr > fpr) or np.all(fnr < fpr)):
            return np.nan
        try:
            tpr_interp = interp1d(fpr, tpr, bounds_error=False, fill_value=(tpr[0], tpr[-1]))
            return brentq(lambda x: 1. - x - tpr_interp(x), 0., 1., maxiter=100)
        except (ValueError, RuntimeError):
            return np.nan
            
    eer = get_eer(y_true, y_scores)
    
    # 10. Brier Score (Lower is better, measures probability accuracy)
    brier = brier_score_loss(y_true, y_scores)
    
    # 11. Expected Calibration Error (Lower is better)
    # ece = calculate_ece(y_true, y_scores)
    
    return pd.Series({
        'accuracy': acc,
        'precision': prec,
        'recall': rec,
        'f1_score': f1,
        'roc_auc': roc,
        'specificity': spec,
        'balanced_accuracy': bal_acc,
        'mcc': mcc,
        'eer': eer,
        'brier_score': brier, # New
        # 'ece': ece            # New
    })

# --- Execution ---
metrics_df = combined_df.groupby(
    ['model', 'seed', 'stratified_split', 'calibrated'], 
    sort=False
).apply(calculate_metrics).reset_index()

print(f"Metrics calculated for {len(metrics_df)} groups.")
# Helper function for IQR
def get_iqr(series):
    return series.quantile(0.75) - series.quantile(0.25)

# --- Aggregation ---
agg_cols = ['model', 'stratified_split', 'calibrated']
metric_cols = [c for c in metrics_df.columns if c not in ['seed'] + agg_cols]

# Define aggregation strategies:
# 1. Metrics: Mean, Std, IQR
# 2. Seed Count: We use 'seed' column to count unique seeds per group
agg_dict = {col: ['mean', 'std', get_iqr] for col in metric_cols}
agg_dict['seed'] = 'count'  # Counts the number of seeds contributing to the group

final_results = metrics_df.groupby(agg_cols).agg(agg_dict).reset_index()

# --- Flatten and Rename Columns ---
final_results.columns = ['_'.join(col).strip() if col[1] else col[0] for col in final_results.columns.values]

new_columns = []
for col in final_results.columns:
    if col.endswith('_mean'):
        new_columns.append(col.replace('_mean', ''))
    elif col.endswith('_std'):
        new_columns.append(col)
    elif col.endswith('_get_iqr'):
        new_columns.append(col.replace('_get_iqr', '_iqr'))
    elif col == 'seed_count':
        new_columns.append('n_seeds') # Rename seed count to n_seeds
    else:
        new_columns.append(col)

final_results.columns = new_columns

# Display results
print("✅ Final Results with N Seeds and IQR:")
display_cols = ['model', 'n_seeds', 'accuracy', 'accuracy_std', 'accuracy_iqr', 'f1_score', 'f1_score_std', 'f1_score_iqr']
print(final_results[display_cols].to_string(index=False))

# Optional: Save
final_results.to_csv("final_evaluation_summary_with_iqr.csv", index=False)


In [ ]:
"""
Cell: Generate Summary Table with 95% CI (t-Student Distribution).
"""

import numpy as np
import pandas as pd
from scipy import stats
import os

# --- CONFIGURATION ---
CI_LEVEL: float = 0.95
OUTPUT_DIR = "../tables"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Define the metrics to include in the summary
METRICS = [
    'accuracy', 'precision', 'recall', 'f1_score',
    'roc_auc', 'specificity', 'balanced_accuracy',
    'mcc', 'eer', 'brier_score', 
    # 'ece'
]

# Prepare data: Select only the necessary columns
cols_needed = ['model', 'stratified_split', 'calibrated', 'n_seeds'] + \
              [f'{m}' for m in METRICS] + \
              [f'{m}_std' for m in METRICS]

# Filter to only existing columns (in case some metrics are missing)
cols_needed = [c for c in cols_needed if c in final_results.columns]
df_summary_input = final_results[cols_needed].copy()

# Drop rows with missing n_seeds or n_seeds < 2 (cannot calculate CI)
df_summary_input = df_summary_input.dropna(subset=['n_seeds'])
df_summary_input = df_summary_input[df_summary_input['n_seeds'] >= 2]

# --- CALCULATE CI ---
# Melt the data to long format for easier processing
id_vars = ['model', 'stratified_split', 'calibrated', 'n_seeds']
value_vars = METRICS

df_melted = df_summary_input.melt(
    id_vars=id_vars,
    value_vars=value_vars,
    var_name='Metric',
    value_name='Mean'
)

# Add std column
df_melted['Std'] = df_melted.apply(
    lambda row: df_summary_input.set_index(
        ['model', 'stratified_split', 'calibrated']
    ).loc[
        (row['model'], row['stratified_split'], row['calibrated']),
        f"{row['Metric']}_std"
    ] if (row['model'], row['stratified_split'], row['calibrated']) in \
    df_summary_input.set_index(['model', 'stratified_split', 'calibrated']).index 
    else np.nan,
    axis=1
)

# Function to calculate CI
def calc_ci(row):
    n = int(row['n_seeds'])
    mean = row['Mean']
    std = row['Std']
    
    if pd.isna(mean) or pd.isna(std) or n < 2:
        return pd.Series({'CI_Low': np.nan, 'CI_High': np.nan, 'CI_Error': np.nan})
    
    sem = std / np.sqrt(n)
    ci_low, ci_high = stats.t.interval(CI_LEVEL, df=n-1, loc=mean, scale=sem)
    ci_error = (ci_high - ci_low) / 2
    
    return pd.Series({'CI_Low': ci_low, 'CI_High': ci_high, 'CI_Error': ci_error})

# Apply CI calculation
ci_results = df_melted.apply(calc_ci, axis=1)
df_melted = pd.concat([df_melted, ci_results], axis=1)

# --- FORMAT OUTPUT ---
# Select and rename columns for clarity
summary_table = df_melted[[
    'model', 'stratified_split', 'calibrated', 'n_seeds',
    'Metric', 'Mean', 'Std', 'CI_Low', 'CI_High', 'CI_Error'
]].copy()

# Rename columns
summary_table.columns = [
    'Model', 'Stratified_Split', 'Calibrated', 'N_Seed',
    'Metric', 'Mean', 'Std', 'CI_95_Low', 'CI_95_High', 'CI_95_Error'
]

# Format CI as string: "mean ± error"
summary_table['CI_95_Formatted'] = summary_table.apply(
    lambda row: f"{row['Mean']:.4f} ± {row['CI_95_Error']:.4f}" 
    if pd.notnull(row['CI_95_Error']) else f"{row['Mean']:.4f}",
    axis=1
)

# Sort for readability
summary_table = summary_table.sort_values(
    by=['Model', 'Stratified_Split', 'Calibrated', 'Metric']
).reset_index(drop=True)

# Display
print("✅ Summary Table with 95% CI (t-Student):")
display_cols = ['Model', 'N_Seed', 'Stratified_Split', 'Calibrated', 
                'Metric', 'Mean', 'CI_95_Error', 'CI_95_Formatted']
print(summary_table[display_cols].to_string(index=False))

# Save to CSV
filename = "summary_metrics_with_ci.csv"
summary_table.to_csv(os.path.join(OUTPUT_DIR, filename), index=False)
print(f"✅ Saved: {filename}")   

print("✅ Summary Table with 95% CI (t-Student):")
display_cols = ['Model', 'N_Seed', 'Stratified_Split', 'Calibrated', 
                'Metric', 'Mean', 'CI_95_Error', 'CI_95_Formatted']
print(summary_table[display_cols].to_string(index=False))   

In [ ]:
summary_table['Metric'].unique()

In [ ]:
# Filter for the specific condition (e.g., Stratified & Calibrated)
mask = (final_results['stratified_split'] == True) & (final_results['calibrated'] == True)
f1_df = final_results[mask][['model', 'n_seeds', 'f1_score', 'f1_score_std', 'f1_score_iqr']].copy()

# Sort by f1_score in descending order (highest to lowest)
f1_df = f1_df.sort_values(by='f1_score', ascending=False).reset_index(drop=True)

# Display
print("✅ F1-Score Sorted (High to Low):")
print(f1_df.to_string(index=False))   

In [ ]:
# 1. Select the necessary columns including split and calibration flags
cols_needed = ['model', 'n_seeds', 'stratified_split', 'calibrated', 
               'f1_score', 'f1_score_std', 'f1_score_iqr']

# 2. NO MASK - Keep all conditions (True/False for split and calibrated)
f1_df = final_results[cols_needed].copy()

# 3. Rename the metric column to include '_mean' suffix
f1_df = f1_df.rename(columns={'f1_score': 'f1_score_mean'})

# 4. Sort by the renamed column in descending order
f1_df = f1_df.sort_values(by='f1_score_mean', ascending=False).reset_index(drop=True)

# 5. Display
print("✅ F1-Score Mean Sorted (High to Low) - All Conditions:")
print(f1_df.to_string(index=False))   

In [ ]:
f1_df = final_results[
    ['model', 'stratified_split', 'calibrated', 'n_seeds',
     'accuracy', 'accuracy_std',
     'balanced_accuracy', 'balanced_accuracy_std',
     'f1_score', 'f1_score_std',
     'mcc', 'mcc_std',
     'precision', 'precision_std',
     'recall', 'recall_std',
     'roc_auc', 'roc_auc_std',
     'specificity', 'specificity_std',
     'eer', 'eer_std']
].copy()

f1_df = f1_df.sort_values(
    by=['stratified_split', 'calibrated', 'model'],
    ascending=[False, False, True]
).reset_index(drop=True)

print(f1_df.to_string(index=False))   

In [ ]:
# ece_brier = final_results[
#     ['model', 'stratified_split', 'calibrated', 'n_seeds',
#      'ece', 'ece_std',
#      'brier_score', 'brier_score_std']
# ].copy()

# ece_brier = ece_brier.sort_values(
#     by=['stratified_split', 'calibrated', 'model'],
#     ascending=[False, False, True]
# ).reset_index(drop=True)

# print(ece_brier.to_string(index=False))

## **Metrics Box-Plot**
____

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import os

# --- CONFIGURATION ---
OUTPUT_DIR = "../plots"
os.makedirs(OUTPUT_DIR, exist_ok=True)

ID_COLUMNS = ['model', 'seed', 'stratified_split', 'calibrated']
METRIC_COLUMNS = [
    'accuracy', 'precision', 'recall', 'f1_score',
    'roc_auc', 'specificity', 'balanced_accuracy',
    'mcc', 'eer', 'brier_score', 
    # 'ece'
]

# Plotting constants (PEP8: avoid magic numbers)
FIGSIZE = (14, 9)
FONT_SIZE_LABEL = 20
FONT_SIZE_TICK = 18
FONT_SIZE_LEGEND = 14
FONT_SIZE_LEGEND_TITLE = 16
GRID_ALPHA = 0.7
GRID_LINEWIDTH = 2.8
GRID_COLOR = "#555555"  # <--- Added: Change to "grey" or "#555555" if preferred
BOX_WIDTH = 0.9
MEAN_MARKER_SIZE = 10
MEAN_MARKER_WIDTH = 1.5
OUTLIER_MARKER_SIZE = 5
YTICK_ROTATION = 45

# Define groups to plot
GROUPS_TO_PLOT = [
    {
        'stratified_split': True,
        'calibrated': True,
        'suffix': '_strat_true_calib_true',
        'title': 'Stratified & Calibrated'
    },
    {
        'stratified_split': False,
        'calibrated': False,
        'suffix': '_strat_false_calib_false',
        'title': 'No Stratification & No Calibration'
    }
]

# Melt data once outside the loop
df_melted = pd.melt(
    metrics_df,
    id_vars=ID_COLUMNS,
    value_vars=METRIC_COLUMNS,
    var_name='Metric',
    value_name='Score'
)

# Drop NaN scores (e.g., from failed ROC-AUC calculations)
df_melted = df_melted.dropna(subset=['Score'])

for group in GROUPS_TO_PLOT:
    mask = (
        (df_melted['stratified_split'] == group['stratified_split']) &
        (df_melted['calibrated'] == group['calibrated'])
    )
    plot_data = df_melted[mask].copy()

    if plot_data.empty:
        print(f"⚠️  No data found for {group['title']}. Skipping.")
        continue

    n_seeds = plot_data['seed'].nunique()
    print(f"Plotting {group['title']} with {n_seeds} seeds...")

    plt.figure(figsize=FIGSIZE)

    ax = sns.boxplot(
        data=plot_data,
        x='Score',
        y='Metric',
        hue='model',
        palette='tab10',
        showmeans=True,
        meanprops={
            "marker": "x",
            "markerfacecolor": "black",
            "markeredgecolor": "black",
            "markersize": MEAN_MARKER_SIZE,
            "markeredgewidth": MEAN_MARKER_WIDTH
        },
        flierprops={
            'marker': 'o',
            'markerfacecolor': 'red',
            'markersize': OUTLIER_MARKER_SIZE,
            'markeredgecolor': 'none'
        },
        width=BOX_WIDTH
    )

    # Custom legend
    mean_handle = plt.Line2D(
        [], [], marker='x', color='black',
        markerfacecolor='black', markeredgecolor='black',
        linestyle='None', markersize=MEAN_MARKER_SIZE,
        markeredgewidth=MEAN_MARKER_WIDTH, label='Mean'
    )
    median_handle = plt.Line2D(
        [], [], color='black', linestyle='-',
        linewidth=2, label='Median'
    )
    outlier_handle = plt.Line2D(
        [], [], marker='o', color='w',
        markerfacecolor='red', markeredgecolor='none',
        linestyle='None', markersize=OUTLIER_MARKER_SIZE,
        label='Outlier'
    )

    handles, labels = ax.get_legend_handles_labels()
    plt.legend(
        handles=handles + [mean_handle, median_handle, outlier_handle],
        loc='upper left',
        bbox_to_anchor=(0, 1),
        frameon=True,
        shadow=True,
        fontsize=FONT_SIZE_LEGEND,
        title_fontsize=FONT_SIZE_LEGEND_TITLE
    )

    # Axis & Grid
    plt.xlabel('Score', fontsize=FONT_SIZE_LABEL, weight='bold')
    plt.ylabel('Metric', fontsize=FONT_SIZE_LABEL, weight='bold')
    plt.xticks(np.arange(0.0, 1.01, 0.1), fontsize=FONT_SIZE_TICK)
    plt.yticks(rotation=YTICK_ROTATION, fontsize=FONT_SIZE_TICK)
    plt.xlim(0.0, 1.0)
    
    # <--- MODIFIED: Added color parameter
    plt.grid(
        axis='x',
        linestyle='--',
        alpha=GRID_ALPHA,
        linewidth=GRID_LINEWIDTH,
        color=GRID_COLOR
    )

    plt.tight_layout()

    # Save
    filename = f"metrics_boxplot{group['suffix']}.svg"
    plt.savefig(
        os.path.join(OUTPUT_DIR, filename),
        format='svg',
        dpi=300,
        bbox_inches='tight'
    )
    print(f"Saved: {filename}")

plt.show()

## **Confidence Interval (t-Student with 1-\alfa = .95)**
____

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

# 1. Prepare Data: Melt the metrics_df
# Ensure we only melt the metric columns, keeping grouping info if needed
# Assuming metrics_df has columns: ['model', 'seed', 'stratified_split', 'calibrated', 'accuracy', 'f1_score', ...]
id_vars = ['model', 'seed', 'stratified_split', 'calibrated']
value_vars = [c for c in metrics_df.columns if c not in id_vars]

df_melted = metrics_df.melt(
    id_vars=id_vars, 
    value_vars=value_vars, 
    var_name='Metric', 
    value_name='Score'
)

# Drop NaNs explicitly (e.g., if a seed failed to calculate ROC-AUC)
# This ensures 'n' in the CI calculation reflects only valid runs
df_melted = df_melted.dropna(subset=['Score'])

# 2. Calculate CI Function
def calculate_ci(group):
    n = len(group)
    if n < 2:
        # Cannot calculate CI with less than 2 samples
        return pd.Series({
            'mean': group.mean(),
            'error': np.nan,
            'ci_low': np.nan,
            'ci_high': np.nan,
            'std': group.std(),
            'count': n
        })
    
    mean = group.mean()
    sem = group.sem()
    
    # 95% CI using t-distribution
    ci_low, ci_high = stats.t.interval(0.95, df=n-1, loc=mean, scale=sem)
    error = (ci_high - ci_low) / 2
    
    return pd.Series({
        'mean': mean,
        'error': error,
        'ci_low': ci_low,
        'ci_high': ci_high,
        'std': group.std(),
        'count': n
    })

# 3. Apply Groupby
# Group by model, config, and metric
ci_table = df_melted.groupby(['model', 'stratified_split', 'calibrated', 'Metric'])['Score'].apply(calculate_ci).unstack()
ci_table = ci_table.reset_index()

# 4. Formatting
cols = ['model', 'stratified_split', 'calibrated', 'Metric', 'mean', 'error', 'ci_low', 'ci_high', 'std', 'count']
final_cols = [c for c in cols if c in ci_table.columns]
ci_table = ci_table[final_cols]

# Add formatted string
ci_table['ci_95'] = ci_table.apply(
    lambda row: f"{row['mean']:.4f} ± {row['error']:.4f}" if pd.notnull(row['error']) else f"{row['mean']:.4f}", 
    axis=1
)

print(ci_table[['model', 'Metric', 'count', 'mean', 'error', 'ci_95']].head(20))   

In [ ]:
ci_table.Metric.unique()

## **Bar chart of mean metrics ± CI (t-Student)**
____

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import os

# --- CONFIGURATION ---
OUTPUT_DIR = "../plots"
os.makedirs(OUTPUT_DIR, exist_ok=True)

FIGSIZE = (12, 6)
BAR_WIDTH_RATIO = 0.8
CAPSIZE = 4
FONT_SIZE_LABEL = 20
FONT_SIZE_LEGEND = 10
FONT_SIZE_TITLE = 14
GRID_ALPHA = 0.5
GRID_COLOR = "black"  # <--- Added: Change to "grey" or "#555555" if preferred
XTICK_ROTATION = 45
Y_LIM = (0, 1.05)
ERROR_LINEWIDTH = 1.5

# Define configurations to plot
CONFIGS = [
    {
        'stratified_split': True,
        'calibrated': True,
        'suffix': '_strat_calib',
        'title': 'Stratified Split & Calibrated'
    },
    {
        'stratified_split': False,
        'calibrated': False,
        'suffix': '_no_strat_no_calib',
        'title': 'No Stratification & No Calibration'
    }
]

# Ensure 'error' and 'mean' are numeric and drop NaNs
ci_table['mean'] = pd.to_numeric(ci_table['mean'], errors='coerce')
ci_table['error'] = pd.to_numeric(ci_table['error'], errors='coerce')
df_clean = ci_table.dropna(subset=['mean', 'error'])

for cfg in CONFIGS:
    # Filter data for current configuration
    mask = (
        (df_clean['stratified_split'] == cfg['stratified_split']) &
        (df_clean['calibrated'] == cfg['calibrated'])
    )
    df_bar = df_clean[mask].copy()

    if df_bar.empty:
        print(f"⚠️  No data for {cfg['title']}. Skipping.")
        continue

    metrics = df_bar['Metric'].unique()
    models = df_bar['model'].unique()
    n_metrics = len(metrics)
    n_models = len(models)

    if n_models == 0 or n_metrics == 0:
        continue

    # Plot
    fig, ax = plt.subplots(figsize=FIGSIZE)
    x = np.arange(n_metrics)
    width = BAR_WIDTH_RATIO / n_models
    colors = sns.color_palette("tab10", n_models)

    for i, model in enumerate(models):
        model_data = df_bar[df_bar['model'] == model]
        
        # Reindex to ensure consistent metric order
        means = model_data.set_index('Metric').reindex(metrics)['mean'].values
        errors = model_data.set_index('Metric').reindex(metrics)['error'].values
        
        offset = (i - (n_models - 1) / 2) * width
        
        ax.bar(
            x + offset,
            means,
            width,
            yerr=errors,
            capsize=CAPSIZE,
            color=colors[i],
            label=model,
            edgecolor='black',
            linewidth=0.5,
            error_kw={'elinewidth': ERROR_LINEWIDTH}
        )

    # Customize
    ax.set_xticks(x)
    ax.set_xticklabels(metrics, rotation=XTICK_ROTATION, ha='right')
    # ax.set_title(cfg['title'], fontsize=FONT_SIZE_TITLE, fontweight='bold', pad=15)
    ax.set_xlabel('Metric', fontsize=FONT_SIZE_LABEL)
    ax.set_ylabel('Score', fontsize=FONT_SIZE_LABEL)
    ax.set_ylim(*Y_LIM)
    ax.legend(title='Model', loc='upper left', fontsize=FONT_SIZE_LEGEND)
    
    # <--- MODIFIED: Added color parameter
    ax.grid(axis='y', linestyle='--', alpha=GRID_ALPHA, color=GRID_COLOR)

    plt.tight_layout()
    
    # Save
    filename = f"bar_metrics_ci{cfg['suffix']}.svg"
    plt.savefig(
        os.path.join(OUTPUT_DIR, filename),
        format='svg',
        dpi=300,
        bbox_inches='tight'
    )
    print(f"✅ Saved: {filename}")
    plt.show()

## **ROC-AUC Plots**
_____

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import roc_curve, auc
import os

# --- CONFIGURATION ---
OUTPUT_DIR = "../plots"
os.makedirs(OUTPUT_DIR, exist_ok=True)

N_ROWS = 2
N_COLS = 4
FIGSIZE_PER_PLOT = (6, 5)
LINEWIDTH_SEED = 1.2
ALPHA_SEED = 0.5
LINEWIDTH_MEAN = 2.5
ALPHA_CI = 0.2
LINEWIDTH_CHANCE = 0.8
ALPHA_CHANCE = 0.5
FONT_SIZE_TITLE = 20
FONT_SIZE_LABEL = 20
FONT_SIZE_LEGEND = 16
GRID_ALPHA = 0.4
AXIS_PAD = 0.02
SAVE_DPI = 300

# Define conditions (rows)
CONDITIONS = [
    {
        'stratified_split': True,
        'calibrated': True,
        'label': 'Stratified & Calibrated'
    },
    {
        'stratified_split': False,
        'calibrated': False,
        'label': 'No Stratification & No Calibration'
    }
]

# Get unique models (columns)
models = combined_df['model'].unique()
n_models = len(models)

# Adjust N_COLS to match actual number of models
if n_models != N_COLS:
    print(f"⚠️  Adjusting layout: Found {n_models} models instead of {N_COLS}.")
    N_COLS = n_models

# Create seed mapping for consistent labeling (seed_1, seed_2, ...)
unique_seeds = combined_df['seed'].unique()
seed_to_label = {seed: f"Seed {i+1}" for i, seed in enumerate(unique_seeds)}

# Common FPR grid for interpolation (for mean curve)
mean_fpr = np.linspace(0, 1, 100)

fig, axes = plt.subplots(
    N_ROWS, N_COLS,
    figsize=(FIGSIZE_PER_PLOT[0] * N_COLS, FIGSIZE_PER_PLOT[1] * N_ROWS),
    sharex='col',
    sharey='row'
)

# Ensure axes is a 2D array
if N_ROWS == 1:
    axes = axes[np.newaxis, :]
if N_COLS == 1:
    axes = axes[:, np.newaxis]

for row_idx, cond in enumerate(CONDITIONS):
    for col_idx, model_name in enumerate(models):
        ax = axes[row_idx, col_idx]

        # Filter data for this specific condition and model
        mask = (
            (combined_df['model'] == model_name) &
            (combined_df['stratified_split'] == cond['stratified_split']) &
            (combined_df['calibrated'] == cond['calibrated'])
        )
        model_data = combined_df[mask]

        if model_data.empty:
            ax.text(
                0.5, 0.5, 'No Data',
                transform=ax.transAxes,
                ha='center', va='center',
                fontsize=FONT_SIZE_LABEL
            )
            ax.set_title(f'{model_name}', fontsize=FONT_SIZE_TITLE, fontweight='bold')
            continue

        seeds = model_data['seed'].unique()
        colors = sns.color_palette("viridis", len(seeds))

        # Store interpolated TPRs for mean curve calculation
        interp_tprs = []

        # Plot individual seed ROC curves
        for s_idx, seed_val in enumerate(seeds):
            run = model_data[model_data['seed'] == seed_val]
            y_true = run['true_label'].values.astype(int)
            y_scores = run['pbb_score'].values.astype(float)

            if len(np.unique(y_true)) < 2:
                continue

            fpr, tpr, _ = roc_curve(y_true, y_scores)
            roc_auc = auc(fpr, tpr)

            # Use mapped seed label (Seed 1, Seed 2, ...)
            seed_label = seed_to_label.get(seed_val, f"Seed {seed_val}")

            ax.plot(
                fpr, tpr,
                color=colors[s_idx],
                linewidth=LINEWIDTH_SEED,
                alpha=ALPHA_SEED,
                label=f'{seed_label} (AUC={roc_auc:.3f})'
            )

            # Interpolate for mean curve
            interp_tpr = np.interp(mean_fpr, fpr, tpr)
            interp_tpr[0] = 0.0
            interp_tprs.append(interp_tpr)

        # Calculate and plot Mean ROC Curve
        if interp_tprs:
            mean_tpr = np.mean(interp_tprs, axis=0)
            mean_tpr[-1] = 1.0
            mean_auc = auc(mean_fpr, mean_tpr)
            std_tpr = np.std(interp_tprs, axis=0)

            ax.plot(
                mean_fpr, mean_tpr,
                color='black',
                linewidth=LINEWIDTH_MEAN,
                label=f'Mean (AUC={mean_auc:.3f})'
            )

            # Plot confidence interval (±1 std)
            tprs_upper = np.minimum(mean_tpr + std_tpr, 1)
            tprs_lower = np.maximum(mean_tpr - std_tpr, 0)
            ax.fill_between(
                mean_fpr, tprs_lower, tprs_upper,
                color='grey', alpha=ALPHA_CI,
                label='±1 std dev'
            )

        # Chance level
        ax.plot(
            [0, 1], [0, 1],
            'k--',
            linewidth=LINEWIDTH_CHANCE,
            alpha=ALPHA_CHANCE
        )

        # Styling
        ax.set_title(
            f'{model_name}',
            fontsize=FONT_SIZE_TITLE,
            fontweight='bold'
        )
        ax.set_xlim([-AXIS_PAD, 1 + AXIS_PAD])
        ax.set_ylim([-AXIS_PAD, 1 + AXIS_PAD])
        ax.legend(
            loc='lower right',
            fontsize=FONT_SIZE_LEGEND,
            framealpha=0.8
        )
        ax.grid(True, linestyle='--', alpha=GRID_ALPHA)

# --- AXIS LABELS ---

# 1. Set Y-axis label ONLY on the first column for each row
for row_idx in range(N_ROWS):
    axes[row_idx, 0].set_ylabel(
        'True Positive Rate (TPR)',
        fontsize=FONT_SIZE_LABEL,
        rotation=90,
        labelpad=15
    )

# 2. Set X-axis label ONLY on the bottom row
for col_idx in range(N_COLS):
    axes[N_ROWS - 1, col_idx].set_xlabel(
        'False Positive Rate (FPR)',
        fontsize=FONT_SIZE_LABEL
    )

# 3. Add Row Headers (Condition Names) on the Left
for row_idx, cond in enumerate(CONDITIONS):
    y_pos = 1 - (row_idx + 0.5) / N_ROWS
    fig.text(
        0.01, y_pos,
        cond['label'],
        va='center',
        ha='center',
        fontsize=FONT_SIZE_LABEL,
        fontweight='bold',
        rotation=90
    )

plt.tight_layout(rect=[0.05, 0, 1, 1])

# Save as PNG (lighter file size than SVG)
filename = "roc_curves_linear_scale.png"
plt.savefig(
    os.path.join(OUTPUT_DIR, filename),
    dpi=SAVE_DPI,
    bbox_inches='tight'
)
print(f"✅ Saved: {filename}")
plt.show()

## **DET Curves**
____

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import det_curve
from scipy.interpolate import interp1d
from scipy.optimize import brentq
import os

# --- CONFIGURATION ---
OUTPUT_DIR = "../plots"
os.makedirs(OUTPUT_DIR, exist_ok=True)

N_ROWS = 2
N_COLS = 4
FIGSIZE_PER_PLOT = (6, 5)
LINEWIDTH_CURVE = 1.5
ALPHA_CURVE = 0.7
MARKERSIZE_EER = 6
FONT_SIZE_TITLE = 20
FONT_SIZE_LABEL = 20
FONT_SIZE_LEGEND = 16
GRID_ALPHA = 0.6
CLIP_EPS = 1e-9
SAVE_DPI = 300

# Define conditions (rows)
CONDITIONS = [
    {
        'stratified_split': True,
        'calibrated': True,
        'label': 'Stratified & Calibrated'
    },
    {
        'stratified_split': False,
        'calibrated': False,
        'label': 'No Stratification & No Calibration'
    }
]

# Get unique models (columns)
models = combined_df['model'].unique()
n_models = len(models)

# Adjust N_COLS to match actual number of models
if n_models != N_COLS:
    print(f"⚠️  Adjusting layout: Found {n_models} models instead of {N_COLS}.")
    N_COLS = n_models

# Create seed mapping for consistent labeling (Seed 1, Seed 2, ...)
unique_seeds = combined_df['seed'].unique()
seed_to_label = {seed: f"Seed {i+1}" for i, seed in enumerate(unique_seeds)}

fig, axes = plt.subplots(
    N_ROWS, N_COLS,
    figsize=(FIGSIZE_PER_PLOT[0] * N_COLS, FIGSIZE_PER_PLOT[1] * N_ROWS),
    sharex='col',
    sharey='row'
)

# Ensure axes is a 2D array
if N_ROWS == 1:
    axes = axes[np.newaxis, :]
if N_COLS == 1:
    axes = axes[:, np.newaxis]

for row_idx, cond in enumerate(CONDITIONS):
    for col_idx, model_name in enumerate(models):
        ax = axes[row_idx, col_idx]

        # Filter data for this specific condition and model
        mask = (
            (combined_df['model'] == model_name) &
            (combined_df['stratified_split'] == cond['stratified_split']) &
            (combined_df['calibrated'] == cond['calibrated'])
        )
        model_data = combined_df[mask]

        if model_data.empty:
            ax.text(
                0.5, 0.5, 'No Data',
                transform=ax.transAxes,
                ha='center', va='center',
                fontsize=FONT_SIZE_LABEL
            )
            ax.set_title(f'{model_name}', fontsize=FONT_SIZE_TITLE, fontweight='bold')
            continue

        seeds = model_data['seed'].unique()
        colors = sns.color_palette("viridis", len(seeds))

        # 1. Set Axis Limits to 0.0 - 1.0 (Linear Probability Scale)
        ax.set_xlim(0.0, 1.0)
        ax.set_ylim(0.0, 1.0)
        
        # Increase tick label font size
        ax.tick_params(axis='both', labelsize=FONT_SIZE_LABEL)

        # 2. Plot Diagonal (EER Line)
        ax.plot([0, 1], [0, 1], 'k--', linewidth=2, label='EER Line', zorder=0)

        # 3. Plot each seed's DET curve
        for s_idx, seed_val in enumerate(seeds):
            run = model_data[model_data['seed'] == seed_val]
            y_true = run['true_label'].values.astype(int)
            y_scores = run['pbb_score'].values.astype(float)

            if len(np.unique(y_true)) < 2:
                continue

            fpr, fnr, _ = det_curve(y_true, y_scores)
            
            # Clip to avoid interpolation errors at exact 0/1
            fpr_safe = np.clip(fpr, CLIP_EPS, 1 - CLIP_EPS)
            fnr_safe = np.clip(fnr, CLIP_EPS, 1 - CLIP_EPS)

            # Use mapped seed label (Seed 1, Seed 2, ...)
            seed_label = seed_to_label.get(seed_val, f"Seed {seed_val}")

            ax.plot(
                fpr_safe, fnr_safe,
                color=colors[s_idx],
                linewidth=LINEWIDTH_CURVE,
                alpha=ALPHA_CURVE,
                label=seed_label
            )

            # Plot EER point
            # try:
            #     fnr_interp = interp1d(fpr_safe, fnr_safe, bounds_error=False,
            #                            fill_value=(fnr_safe[0], fnr_safe[-1]))
            #     eer_val = brentq(
            #         lambda x: x - fnr_interp(x), 0, 1, maxiter=100
            #     )
            #     ax.plot(
            #         eer_val, eer_val, 'o',
            #         color=colors[s_idx],
            #         markersize=MARKERSIZE_EER,
            #         markeredgecolor='black',
            #         zorder=5
            #     )
            # except (ValueError, RuntimeError):
            #     pass

        # 5. Customize subplot
        ax.set_title(f'{model_name}', fontsize=FONT_SIZE_TITLE, fontweight='bold')
        ax.grid(True, linestyle='--', alpha=GRID_ALPHA)
        
        # Show legend on all subplots for clarity
        ax.legend(loc='upper right', fontsize=FONT_SIZE_LEGEND, framealpha=0.8)

# --- AXIS LABELS ---

# 1. Set Y-axis label to "FNR" on the first column for each row
for row_idx in range(N_ROWS):
    ax_left = axes[row_idx, 0]
    ax_left.set_ylabel(
        'False Negative Rate (FNR)', 
        fontsize=FONT_SIZE_LABEL, 
        rotation=90, 
        labelpad=15
    )

# 2. Set X-axis label to "FPR" on the bottom row
for col_idx in range(N_COLS):
    ax_bottom = axes[N_ROWS - 1, col_idx]
    ax_bottom.set_xlabel(
        'False Positive Rate (FPR)', 
        fontsize=FONT_SIZE_LABEL
    )

# 3. Add Row Headers (Condition Names) on the Left
for row_idx, cond in enumerate(CONDITIONS):
    y_pos = 1 - (row_idx + 0.5) / N_ROWS
    fig.text(
        0.01, y_pos, 
        cond['label'],
        va='center', 
        ha='center',
        fontsize=FONT_SIZE_LABEL,
        fontweight='bold',
        rotation=90
    )

plt.tight_layout(rect=[0.05, 0, 1, 1])

# Save as PNG (lighter file size than SVG)
# filename = "det_curves_linear_scale.png"
# plt.savefig(
#     os.path.join(OUTPUT_DIR, filename),
#     dpi=SAVE_DPI,
#     bbox_inches='tight'
# )
# print(f"✅ Saved: {filename}")
# plt.show()   

# Save Vectorized SVG
filename = "det_curves_linear_scale.svg"
plt.savefig(
    os.path.join(OUTPUT_DIR, filename),
    format='svg',
    dpi=300,
    bbox_inches='tight'
)
print(f"✅ Saved: {filename}")
plt.show()

## **Reliability Diagrams**
____

| Definition | Y-axis | Formal Statement | Source |
|------------|--------|-----------------|--------|
| **Confidence Calibration** (Guo et al., 2017; Kull et al., 2019) | **Accuracy** = `P(Y = ŷ \| p̂ = p)` (fraction of *correct predictions*) | "When I am *p* confident in my *predicted class*, am I correct *p*% of the time?" | Guo et al. :inlineCitations{data="&#91;&#123;&quot;url&quot;&#58;&quot;https&#58;//mbrenndoerfer.com/writing/calibration-machine-learning-confidence-accuracy-ece&quot;,&quot;favicon&quot;&#58;&quot;https&#58;//imgs.search.brave.com/AYZEHDnjRFBV_hAMEMrHk_wcOrJMqSwjoVV9TYvlisA/rs&#58;fit&#58;32&#58;32&#58;1&#58;0/g&#58;ce/aHR0cDovL2Zhdmlj/b25zLnNlYXJjaC5i/cmF2ZS5jb20vaWNv/bnMvZGU1ZTAxZGEw/MWJiNTExNGQ2M2Fi/NTRiMjJkMzQ5Yjdk/Nzg5Zjc0OTkwZDlk/YmRmNjYyY2NkMmJk/M2JmNWYxOC9tYnJl/bm5kb2VyZmVyLmNv/bS8&quot;,&quot;title&quot;&#58;&quot;Calibration&#32;in&#32;Machine&#32;Learning&#58;&#32;Confidence,&#32;Accuracy&#32;&amp;&#32;ECE&#32;-&#32;...&quot;,&quot;snippet&quot;&#58;&quot;##&#32;SummaryLink&#32;Copied&#92;nCalibration&#32;measures&#32;the&#32;alignment&#32;between&#32;a&#32;model's&#32;predicted&#32;confidence&#32;and&#32;its&#32;actual&#32;accuracy.&#32;A&#32;perfectly&#32;calibrated&#32;model&#32;satisfies&#32;the&#32;condition&#32;that&#32;among&#32;all&#32;predictions&#32;where&#32;the&#32;model&#32;claims&#32;confidence&#32;,&#32;exactly&#32;fraction&#32;are&#32;correct.&quot;&#125;,&#123;&quot;url&quot;&#58;&quot;https&#58;//medium.com/analytics-vidhya/how-probability-calibration-works-a4ba3f73fd4d&quot;,&quot;favicon&quot;&#58;&quot;https&#58;//imgs.search.brave.com/4R4hFITz_F_be0roUiWbTZKhsywr3fnLTMTkFL5HFow/rs&#58;fit&#58;32&#58;32&#58;1&#58;0/g&#58;ce/aHR0cDovL2Zhdmlj/b25zLnNlYXJjaC5i/cmF2ZS5jb20vaWNv/bnMvOTZhYmQ1N2Q4/NDg4ZDcyODIyMDZi/MzFmOWNhNjE3Y2E4/Y2YzMThjNjljNDIx/ZjllZmNhYTcwODhl/YTcwNDEzYy9tZWRp/dW0uY29tLw&quot;,&quot;title&quot;&#58;&quot;How&#32;Probability&#32;Calibration&#32;Works&quot;,&quot;snippet&quot;&#58;&quot;This&#32;can&#32;be&#32;visualized&#32;with&#32;the&#32;following&#32;plot&#58;&#92;n&#92;n*The&#32;perfectly&#32;calibrated&#32;line&#32;of&#32;an&#32;ideal&#32;model.&#32;A&#32;model&#32;is&#32;perfectly&#32;calibrated&#32;if,&#32;for&#32;any&#32;p,&#32;a&#32;prediction&#32;of&#32;a&#32;class&#32;with&#32;confidence&#32;p&#32;is&#32;correct&#32;100*p%&#32;of&#32;the&#32;time.&#91;…&quot;&#125;&#93;"} 2017; Nixon et al. 2019; Pavlovic 2025 |
| **Classwise / Probability Calibration** (Wilks, 1995; Naeini et al., 2015) | **Fraction of positives** = `P(Y = 1 \| p̂ = p)` (prevalence of the positive class) | "When I predict probability *p* for the *positive class*, is the positive class present *p*% of the time?" | Wilks 1995; scikit-learn; Naeini et al. 2015 |

### **Based on Confidence Calibration** (https://arxiv.org/abs/2501.19047v2)
____

In [ ]:
"""
Cell 1: Configuration and Data Filtering.
Defines target parameters and filters the dataframe before metric calculation.
"""

# --- TARGET CONFIGURATION ---
# Set these to your desired values before running Cell 2
TARGET_MODEL: str = "xgboost"
TARGET_SEED: str = combined_df["seed"].unique()[0]  # Change to desired seed
TARGET_STRATIFIED: bool = True
TARGET_CALIBRATED: bool = True
USE_MEAN: bool = True  # If True, aggregates all seeds for the model/config

# --- FILTER DATA ---
# Base mask: model + stratified + calibrated
base_mask = (
    (combined_df["model"] == TARGET_MODEL) &
    (combined_df["stratified_split"] == TARGET_STRATIFIED) &
    (combined_df["calibrated"] == TARGET_CALIBRATED)
)

if USE_MEAN:
    # Aggregate all seeds for this model/config
    run_data = combined_df[base_mask].copy()
    data_label = f"Mean (All Seeds)"
else:
    # Filter to specific seed
    run_data = combined_df[base_mask & (combined_df["seed"] == TARGET_SEED)].copy()
    data_label = f"Seed {TARGET_SEED}"

if run_data.empty:
    raise ValueError(
        f"No data found for model={TARGET_MODEL}, "
        f"stratified={TARGET_STRATIFIED}, calibrated={TARGET_CALIBRATED}, "
        f"seed={TARGET_SEED if not USE_MEAN else 'ALL'}"
    )

print(f"✅ Filtered {len(run_data)} samples.")
print(f"   Model: {TARGET_MODEL}")
print(f"   Stratified: {TARGET_STRATIFIED}")
print(f"   Calibrated: {TARGET_CALIBRATED}")
print(f"   Data: {data_label}")

"""
Cell 2: Compute binned calibration metrics from filtered data.
"""

import numpy as np
from typing import Dict, Any

# --- CONSTANTS ---
NUM_BINS: int = 10


def compute_calibration(
    true_labels: np.ndarray,
    pred_labels: np.ndarray,
    confidences: np.ndarray,
    num_bins: int = NUM_BINS,
) -> Dict[str, Any]:
    """Compute binned calibration metrics.

    Args:
        true_labels: True binary labels.
        pred_labels: Predicted binary labels.
        confidences: Predicted probabilities in [0, 1].
        num_bins: Number of confidence bins.

    Returns:
        Dictionary with bin accuracies, confidences, counts, ECE, MCE.

    Raises:
        ValueError: If arrays mismatch or no data in bins.
    """
    if not (len(confidences) == len(pred_labels) == len(true_labels)):
        raise ValueError("All arrays must have the same length.")
    if num_bins <= 0:
        raise ValueError("num_bins must be positive.")

    bins = np.linspace(0.0, 1.0, num_bins + 1)
    indices = np.digitize(confidences, bins, right=True)

    bin_accs = np.zeros(num_bins, dtype=np.float64)
    bin_confs = np.zeros(num_bins, dtype=np.float64)
    bin_counts = np.zeros(num_bins, dtype=np.int64)

    for b in range(num_bins):
        sel = np.where(indices == b + 1)[0]
        if len(sel) > 0:
            bin_accs[b] = np.mean(true_labels[sel] == pred_labels[sel])
            bin_confs[b] = np.mean(confidences[sel])
            bin_counts[b] = len(sel)

    total = np.sum(bin_counts)
    if total == 0:
        raise ValueError("No data points found in any bin.")

    gaps = np.abs(bin_accs - bin_confs)
    ece = np.sum(gaps * bin_counts) / total
    mce = np.max(gaps)

    return {
        "accuracies": bin_accs,
        "confidences": bin_confs,
        "counts": bin_counts,
        "bins": bins,
        "avg_accuracy": np.sum(bin_accs * bin_counts) / total,
        "avg_confidence": np.sum(bin_confs * bin_counts) / total,
        "expected_calibration_error": ece,
        "max_calibration_error": mce,
    }


# --- COMPUTE ---
y_true = run_data["true_label"].values.astype(int)
y_pred = run_data["predicted_label"].values.astype(int)
y_prob = run_data["pbb_score"].values.astype(float)

if len(np.unique(y_true)) < 2:
    raise ValueError("Single class in test set. Cannot compute calibration.")

bin_data = compute_calibration(y_true, y_pred, y_prob, NUM_BINS)

print(f"✅ Calibration computed.")
print(f"   ECE: {bin_data['expected_calibration_error']:.4f}")
print(f"   MCE: {bin_data['max_calibration_error']:.4f}")
print(f"   Avg Accuracy: {bin_data['avg_accuracy']:.4f}")
print(f"   Avg Confidence: {bin_data['avg_confidence']:.4f}")   

"""
Cell 3: Plot reliability diagram and confidence histogram.
"""

import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, Any

# --- PLOT CONSTANTS ---
BAR_COLOR_RGB: tuple = (240 / 255, 60 / 255, 60 / 255)
BAR_ALPHA: float = 0.3
ACC_LINE_COLOR: str = "black"
DIAG_COLOR: str = "gray"
LINEWIDTH_BAR: float = 1
LINEWIDTH_ACC: float = 3
LINEWIDTH_DIAG: float = 1.5
FONT_SIZE_ECE: int = 12
FONT_SIZE_TITLE: int = 14
FONT_SIZE_LABEL: int = 12
FONT_SIZE_LEGEND: int = 10
HEIGHT_RATIO_DIAG: int = 4
HEIGHT_RATIO_HIST: int = 1
HIST_WIDTH_FACTOR: float = 0.9
HSPACE_COMBINED: float = -0.1
FIGSIZE: tuple = (6, 8.4)
DPI: int = 300

# Build a safe filename from the current configuration
seed_part = "mean" if USE_MEAN else f"seed_{TARGET_SEED}"
filename = (
    f"reliability_{TARGET_MODEL}_{seed_part}"
    f"_strat_{str(TARGET_STRATIFIED).lower()}"
    f"_calib_{str(TARGET_CALIBRATED).lower()}.svg"
)


def plot_reliability_with_hist(
    ax_top: plt.Axes,
    ax_bot: plt.Axes,
    bin_data: Dict[str, Any],
    title: str = "",
    draw_ece: bool = True,
    draw_averages: bool = True,
) -> None:
    """Plot reliability diagram (top) and confidence histogram (bottom).

    Args:
        ax_top: Axes for the reliability diagram.
        ax_bot: Axes for the confidence histogram.
        bin_data: Output from compute_calibration().
        title: Title for the top subplot.
        draw_ece: Show ECE text.
        draw_averages: Show avg lines in histogram.
    """
    accuracies = bin_data["accuracies"]
    confidences = bin_data["confidences"]
    counts = bin_data["counts"]
    bins = bin_data["bins"]

    bin_size = 1.0 / len(counts)
    positions = bins[:-1] + bin_size / 2.0
    widths = bin_size * 0.9

    # --- TOP: Reliability Diagram ---
    gap_bars = ax_top.bar(
        positions,
        np.abs(accuracies - confidences),
        bottom=np.minimum(accuracies, confidences),
        width=widths,
        color=BAR_COLOR_RGB,
        alpha=BAR_ALPHA,
        edgecolor=BAR_COLOR_RGB,
        linewidth=LINEWIDTH_BAR,
        label="Gap",
    )

    acc_bars = ax_top.bar(
        positions,
        0,
        bottom=accuracies,
        width=widths,
        edgecolor=ACC_LINE_COLOR,
        color=ACC_LINE_COLOR,
        alpha=1.0,
        linewidth=LINEWIDTH_ACC,
        label="Accuracy",
    )

    ax_top.plot(
        [0, 1], [0, 1],
        linestyle="--",
        color=DIAG_COLOR,
        linewidth=LINEWIDTH_DIAG,
    )

    if draw_ece:
        ece_pct = bin_data["expected_calibration_error"] * 100
        ax_top.text(
            0.98, 0.02,
            f"ECE={ece_pct:.2f}%",
            color="black",
            ha="right",
            va="bottom",
            transform=ax_top.transAxes,
            fontsize=FONT_SIZE_ECE,
        )

    ax_top.set_xlim(0, 1)
    ax_top.set_ylim(0, 1)
    ax_top.set_aspect("equal")
    ax_top.set_title(title, fontsize=FONT_SIZE_TITLE, fontweight="bold")
    ax_top.set_xlabel("")
    ax_top.set_ylabel("Expected Accuracy", fontsize=FONT_SIZE_LABEL)

    ax_top.legend(
        handles=[gap_bars, acc_bars],
        labels=["Gap", "Accuracy"],
        loc="upper right",
        fontsize=FONT_SIZE_LEGEND,
        framealpha=0.8,
    )

    # --- BOTTOM: Confidence Histogram (inverted) ---
    ax_bot.bar(
        positions,
        -counts,
        width=bin_size * HIST_WIDTH_FACTOR,
        color="steelblue",
        alpha=0.6,
        edgecolor="steelblue",
    )

    ax_bot.set_xlim(0, 1)
    ax_bot.set_xlabel("Confidence", fontsize=FONT_SIZE_LABEL)
    ax_bot.set_ylabel("Count", fontsize=FONT_SIZE_LABEL)

    yticks = ax_bot.get_yticks()
    ax_bot.set_yticklabels([str(int(abs(t))) for t in yticks])

    if draw_averages:
        ax_bot.axvline(
            x=bin_data["avg_accuracy"],
            ls="solid",
            lw=3,
            c="black",
            label="Accuracy",
        )
        ax_bot.axvline(
            x=bin_data["avg_confidence"],
            ls="dotted",
            lw=3,
            c="#444",
            label="Avg. Confidence",
        )
        ax_bot.legend(loc="upper right", fontsize=FONT_SIZE_LEGEND, framealpha=0.8)


# --- PLOT ---
title = (
    f"{TARGET_MODEL} | {data_label}\n"
    f"Stratified: {TARGET_STRATIFIED} | Calibrated: {TARGET_CALIBRATED}"
)

fig, (ax_top, ax_bot) = plt.subplots(
    nrows=2,
    ncols=1,
    sharex=True,
    figsize=FIGSIZE,
    dpi=DPI,
    gridspec_kw={"height_ratios": [HEIGHT_RATIO_DIAG, HEIGHT_RATIO_HIST]},
)

plot_reliability_with_hist(
    ax_top=ax_top,
    ax_bot=ax_bot,
    bin_data=bin_data,
    title=title,
    draw_ece=True,
    draw_averages=True,
)

plt.tight_layout()
plt.subplots_adjust(hspace=HSPACE_COMBINED)

plt.savefig(
    os.path.join(OUTPUT_DIR, filename),
    format="svg",
    dpi=DPI,
    bbox_inches="tight",
)
print(f"✅ Saved: {filename}")
plt.show()   

In [ ]:
"""
Cell 1: Compute binned calibration metrics (Binary-ECE and Confidence-ECE).
"""

import numpy as np
from typing import Dict, Any

# --- CONSTANTS ---
NUM_BINS: int = 10


def compute_calibration_dual(
    true_labels: np.ndarray,
    pred_labels: np.ndarray,
    confidences: np.ndarray,
    num_bins: int = NUM_BINS,
) -> Dict[str, Any]:
    """Compute binned calibration metrics for both ECE definitions.

    Args:
        true_labels: True binary labels.
        pred_labels: Predicted binary labels.
        confidences: Predicted confidences in [0, 1].
        num_bins: Number of confidence bins.

    Returns:
        Dictionary with bin stats, Binary-ECE, and Confidence-ECE.
    """
    if not (len(confidences) == len(pred_labels) == len(true_labels)):
        raise ValueError("All arrays must have the same length.")
    if num_bins <= 0:
        raise ValueError("num_bins must be positive.")

    bins = np.linspace(0.0, 1.0, num_bins + 1)
    indices = np.digitize(confidences, bins, right=True)

    bin_counts = np.zeros(num_bins, dtype=np.int64)
    bin_confs = np.zeros(num_bins, dtype=np.float64)
    
    # For Binary-ECE: fraction of positives
    bin_pos_freq = np.zeros(num_bins, dtype=np.float64)
    
    # For Confidence-ECE: fraction of correct predictions
    bin_acc = np.zeros(num_bins, dtype=np.float64)

    for b in range(num_bins):
        sel = np.where(indices == b + 1)[0]
        if len(sel) > 0:
            bin_counts[b] = len(sel)
            bin_confs[b] = np.mean(confidences[sel])
            bin_pos_freq[b] = np.mean(true_labels[sel])
            bin_acc[b] = np.mean(true_labels[sel] == pred_labels[sel])

    total = np.sum(bin_counts)
    if total == 0:
        raise ValueError("No data points found in any bin.")

    # Calculate Binary-ECE (Fraction of Positives)
    gaps_binary = np.abs(bin_pos_freq - bin_confs)
    binary_ece = np.sum(gaps_binary * bin_counts) / total

    # Calculate Confidence-ECE (Prediction Accuracy)
    gaps_conf = np.abs(bin_acc - bin_confs)
    confidence_ece = np.sum(gaps_conf * bin_counts) / total

    return {
        "accuracies": bin_acc,  # For plotting Confidence-ECE bars
        "pos_freq": bin_pos_freq,  # For plotting Binary-ECE bars
        "confidences": bin_confs,
        "counts": bin_counts,
        "bins": bins,
        "binary_ece": binary_ece,
        "confidence_ece": confidence_ece,
        "avg_accuracy": np.sum(bin_acc * bin_counts) / total,
        "avg_confidence": np.sum(bin_confs * bin_counts) / total,
    }


# --- COMPUTE ---
y_true = run_data["true_label"].values.astype(int)
y_pred = run_data["predicted_label"].values.astype(int)
y_prob = run_data["pbb_score"].values.astype(float)

if len(np.unique(y_true)) < 2:
    raise ValueError("Single class in test set. Cannot compute calibration.")

bin_data = compute_calibration_dual(y_true, y_pred, y_prob, NUM_BINS)

print(f"✅ Calibration computed.")
print(f"   Binary-ECE (Pos Freq):   {bin_data['binary_ece']:.4f}")
print(f"   Confidence-ECE (Acc):    {bin_data['confidence_ece']:.4f}")
print(f"   Avg Accuracy:            {bin_data['avg_accuracy']:.4f}")
print(f"   Avg Confidence:          {bin_data['avg_confidence']:.4f}")

In [ ]:
"""
Cell 2: Plot reliability diagram with Confidence-ECE focus.
"""

import matplotlib.pyplot as plt
import os
from typing import Dict, Any

# --- PLOT CONSTANTS ---
BAR_COLOR_RGB: tuple = (240 / 255, 60 / 255, 60 / 255)
BAR_ALPHA: float = 0.3
ACC_LINE_COLOR: str = "black"
DIAG_COLOR: str = "gray"
LINEWIDTH_BAR: float = 1
LINEWIDTH_ACC: float = 3
LINEWIDTH_DIAG: float = 1.5
FONT_SIZE_ECE: int = 10
FONT_SIZE_TITLE: int = 14
FONT_SIZE_LABEL: int = 12
FONT_SIZE_LEGEND: int = 10
HEIGHT_RATIO_DIAG: int = 4
HEIGHT_RATIO_HIST: int = 1
HIST_WIDTH_FACTOR: float = 0.9
HSPACE_COMBINED: float = -0.1
FIGSIZE: tuple = (6, 8.4)
DPI: int = 300
OUTPUT_DIR = "../plots"


def plot_reliability_confidence(
    ax_top: plt.Axes,
    ax_bot: plt.Axes,
    bin_data: Dict[str, Any],
    title: str = "",
) -> None:
    """Plot reliability diagram (Confidence-ECE) and histogram.

    Args:
        ax_top: Axes for the reliability diagram.
        ax_bot: Axes for the confidence histogram.
        bin_data: Output from compute_calibration_dual().
        title: Title for the top subplot.
    """
    accuracies = bin_data["accuracies"]  # Prediction accuracy per bin
    confidences = bin_data["confidences"]
    counts = bin_data["counts"]
    bins = bin_data["bins"]

    bin_size = 1.0 / len(counts)
    positions = bins[:-1] + bin_size / 2.0
    widths = bin_size * 0.9

    # --- TOP: Reliability Diagram (Confidence-ECE) ---
    # Gap bars: |Accuracy - Confidence|
    gap_bars = ax_top.bar(
        positions,
        np.abs(accuracies - confidences),
        bottom=np.minimum(accuracies, confidences),
        width=widths,
        color=BAR_COLOR_RGB,
        alpha=BAR_ALPHA,
        edgecolor=BAR_COLOR_RGB,
        linewidth=LINEWIDTH_BAR,
        label="Gap (Acc - Conf)",
    )

    # Accuracy line (black)
    acc_bars = ax_top.bar(
        positions,
        0,
        bottom=accuracies,
        width=widths,
        edgecolor=ACC_LINE_COLOR,
        color=ACC_LINE_COLOR,
        alpha=1.0,
        linewidth=LINEWIDTH_ACC,
        label="Accuracy",
    )

    # Diagonal
    ax_top.plot(
        [0, 1], [0, 1],
        linestyle="--",
        color=DIAG_COLOR,
        linewidth=LINEWIDTH_DIAG,
    )

    # Display BOTH ECE values
    # conf_ece_pct = bin_data["confidence_ece"] * 100
    # bin_ece_pct = bin_data["binary_ece"] * 100
    # ax_top.text(
    #     0.98, 0.02,
    #     f"Conf-ECE={conf_ece_pct:.2f}%\nBin-ECE={bin_ece_pct:.2f}%",
    #     color="black",
    #     ha="right",
    #     va="bottom",
    #     transform=ax_top.transAxes,
    #     fontsize=FONT_SIZE_ECE,
    # )

        # Display ONLY Confidence-ECE
    conf_ece_pct = bin_data["confidence_ece"] * 100
    ax_top.text(
        0.98, 0.02,
        f"ECE={conf_ece_pct:.2f}%",
        color="black",
        ha="right",
        va="bottom",
        transform=ax_top.transAxes,
        fontsize=FONT_SIZE_ECE,
    )

    ax_top.set_xlim(0, 1)
    ax_top.set_ylim(0, 1)
    ax_top.set_aspect("equal")
    ax_top.set_title(title, fontsize=FONT_SIZE_TITLE, fontweight="bold")
    ax_top.set_xlabel("")
    ax_top.set_ylabel("Expected Accuracy", fontsize=FONT_SIZE_LABEL)

    ax_top.legend(
        handles=[gap_bars, acc_bars],
        labels=["Gap", "Accuracy"],
        loc="upper right",
        fontsize=FONT_SIZE_LEGEND,
        framealpha=0.8,
    )

    # --- BOTTOM: Confidence Histogram (inverted) ---
    ax_bot.bar(
        positions,
        -counts,
        width=bin_size * HIST_WIDTH_FACTOR,
        color="steelblue",
        alpha=0.6,
        edgecolor="steelblue",
    )

    ax_bot.set_xlim(0, 1)
    ax_bot.set_xlabel("Confidence", fontsize=FONT_SIZE_LABEL)
    ax_bot.set_ylabel("Count", fontsize=FONT_SIZE_LABEL)

    yticks = ax_bot.get_yticks()
    ax_bot.set_yticklabels([str(int(abs(t))) for t in yticks])

    # Two vertical lines: Avg Accuracy & Avg Confidence
    ax_bot.axvline(
        x=bin_data["avg_accuracy"],
        ls="solid",
        lw=2,
        c="black",
        label="Avg Accuracy",
    )
    ax_bot.axvline(
        x=bin_data["avg_confidence"],
        ls="dotted",
        lw=2,
        c="#444",
        label="Avg Confidence",
    )
    ax_bot.legend(loc="upper right", fontsize=FONT_SIZE_LEGEND, framealpha=0.8)


# --- PLOT ---
title = (
    # f"{TARGET_MODEL} | {data_label}\n"
    f"{TARGET_MODEL}\n"
    f"Stratified: {TARGET_STRATIFIED} | Calibrated: {TARGET_CALIBRATED}"
)

fig, (ax_top, ax_bot) = plt.subplots(
    nrows=2,
    ncols=1,
    sharex=True,
    figsize=FIGSIZE,
    dpi=DPI,
    gridspec_kw={"height_ratios": [HEIGHT_RATIO_DIAG, HEIGHT_RATIO_HIST]},
)

plot_reliability_confidence(
    ax_top=ax_top,
    ax_bot=ax_bot,
    bin_data=bin_data,
    title=title,
)

plt.tight_layout()
plt.subplots_adjust(hspace=HSPACE_COMBINED)

# --- SAVE AS SVG ---
os.makedirs(OUTPUT_DIR, exist_ok=True)
seed_part = "mean" if USE_MEAN else f"seed_{TARGET_SEED}"
filename = (
    f"reliability_conf_{TARGET_MODEL}_{seed_part}"
    f"_strat_{str(TARGET_STRATIFIED).lower()}"
    f"_calib_{str(TARGET_CALIBRATED).lower()}.svg"
)

plt.savefig(
    os.path.join(OUTPUT_DIR, filename),
    format="svg",
    dpi=DPI,
    bbox_inches="tight",
)
print(f"✅ Saved: {filename}")
plt.show()   


## **Based on Pbb Calibration**
____

In [ ]:
"""
Cell 1: Configuration and Data Filtering.
"""

# --- TARGET CONFIGURATION ---
TARGET_MODEL: str = "mlp"  # Change to desired model
TARGET_SEED: str = combined_df["seed"].unique()[0]  # Change to desired seed
TARGET_STRATIFIED: bool = True
TARGET_CALIBRATED: bool = True
USE_MEAN: bool = True  # If True, aggregates all seeds for the model/config

# --- FILTER DATA ---
base_mask = (
    (combined_df["model"] == TARGET_MODEL) &
    (combined_df["stratified_split"] == TARGET_STRATIFIED) &
    (combined_df["calibrated"] == TARGET_CALIBRATED)
)

if USE_MEAN:
    run_data = combined_df[base_mask].copy()
    data_label = f"Mean (All Seeds)"
else:
    run_data = combined_df[base_mask & (combined_df["seed"] == TARGET_SEED)].copy()
    data_label = f"Seed {TARGET_SEED}"

if run_data.empty:
    raise ValueError(
        f"No data found for model={TARGET_MODEL}, "
        f"stratified={TARGET_STRATIFIED}, calibrated={TARGET_CALIBRATED}, "
        f"seed={TARGET_SEED if not USE_MEAN else 'ALL'}"
    )

print(f"✅ Filtered {len(run_data)} samples.")
print(f"   Model: {TARGET_MODEL}")
print(f"   Stratified: {TARGET_STRATIFIED}")
print(f"   Calibrated: {TARGET_CALIBRATED}")
print(f"   Data: {data_label}")

In [ ]:
"""
Cell 2: Compute binned calibration metrics using the 'calculate_ece' logic.
"""

import numpy as np
from typing import Dict, Any

# --- CONSTANTS ---
NUM_BINS: int = 10


def compute_binned_calibration(
    true_labels: np.ndarray,
    confidences: np.ndarray,
    num_bins: int = NUM_BINS,
) -> Dict[str, Any]:
    """Compute binned calibration metrics and ECE.

    Args:
        true_labels: True binary labels.
        confidences: Predicted probabilities in [0, 1].
        num_bins: Number of confidence bins.

    Returns:
        Dictionary with bin accuracies, confidences, counts, and ECE.
    """
    if len(confidences) != len(true_labels):
        raise ValueError("Labels and confidences must have the same length.")
    if num_bins <= 0:
        raise ValueError("num_bins must be positive.")

    bins = np.linspace(0.0, 1.0, num_bins + 1)
    indices = np.digitize(confidences, bins, right=True)

    bin_accs = np.zeros(num_bins, dtype=np.float64)
    bin_confs = np.zeros(num_bins, dtype=np.float64)
    bin_counts = np.zeros(num_bins, dtype=np.int64)

    for b in range(num_bins):
        sel = np.where(indices == b + 1)[0]
        if len(sel) > 0:
            bin_accs[b] = np.mean(true_labels[sel])
            bin_confs[b] = np.mean(confidences[sel])
            bin_counts[b] = len(sel)

    total = np.sum(bin_counts)
    if total == 0:
        raise ValueError("No data points found in any bin.")

    # Calculate Binned ECE (same logic as your metrics_df)
    ece = 0.0
    for i in range(num_bins):
        prop_in_bin = bin_counts[i] / total
        if prop_in_bin > 0:
            ece += np.abs(bin_accs[i] - bin_confs[i]) * prop_in_bin

    return {
        "accuracies": bin_accs,
        "confidences": bin_confs,
        "counts": bin_counts,
        "bins": bins,
        "ece": ece,
    }


# --- COMPUTE ---
y_true = run_data["true_label"].values.astype(int)
y_prob = run_data["pbb_score"].values.astype(float)

if len(np.unique(y_true)) < 2:
    raise ValueError("Single class in test set. Cannot compute calibration.")

bin_data = compute_binned_calibration(y_true, y_prob, NUM_BINS)

print(f"✅ Binned Calibration computed.")
print(f"   Binned ECE: {bin_data['ece']:.4f}")   

In [ ]:
"""
Cell 3: Plot reliability diagram and confidence histogram with Binned ECE.
"""

import matplotlib.pyplot as plt
import os
from typing import Dict, Any

# --- PLOT CONSTANTS ---
BAR_COLOR_RGB: tuple = (240 / 255, 60 / 255, 60 / 255)
BAR_ALPHA: float = 0.3
ACC_LINE_COLOR: str = "black"
DIAG_COLOR: str = "gray"
LINEWIDTH_BAR: float = 1
LINEWIDTH_ACC: float = 3
LINEWIDTH_DIAG: float = 1.5
FONT_SIZE_ECE: int = 12
FONT_SIZE_TITLE: int = 14
FONT_SIZE_LABEL: int = 12
FONT_SIZE_LEGEND: int = 10
HEIGHT_RATIO_DIAG: int = 4
HEIGHT_RATIO_HIST: int = 1
HIST_WIDTH_FACTOR: float = 0.9
HSPACE_COMBINED: float = -0.1
FIGSIZE: tuple = (6, 8.4)
DPI: int = 300
OUTPUT_DIR = "../plots"


def plot_reliability_binned(
    ax_top: plt.Axes,
    ax_bot: plt.Axes,
    bin_data: Dict[str, Any],
    title: str = "",
) -> None:
    """Plot reliability diagram (top) and confidence histogram (bottom).

    Args:
        ax_top: Axes for the reliability diagram.
        ax_bot: Axes for the confidence histogram.
        bin_data: Output from compute_binned_calibration().
        title: Title for the top subplot.
    """
    accuracies = bin_data["accuracies"]
    confidences = bin_data["confidences"]
    counts = bin_data["counts"]
    bins = bin_data["bins"]

    bin_size = 1.0 / len(counts)
    positions = bins[:-1] + bin_size / 2.0
    widths = bin_size * 0.9

    # --- TOP: Reliability Diagram ---
    gap_bars = ax_top.bar(
        positions,
        np.abs(accuracies - confidences),
        bottom=np.minimum(accuracies, confidences),
        width=widths,
        color=BAR_COLOR_RGB,
        alpha=BAR_ALPHA,
        edgecolor=BAR_COLOR_RGB,
        linewidth=LINEWIDTH_BAR,
        label="Gap",
    )

    acc_bars = ax_top.bar(
        positions,
        0,
        bottom=accuracies,
        width=widths,
        edgecolor=ACC_LINE_COLOR,
        color=ACC_LINE_COLOR,
        alpha=1.0,
        linewidth=LINEWIDTH_ACC,
        label="Accuracy",
    )

    ax_top.plot(
        [0, 1], [0, 1],
        linestyle="--",
        color=DIAG_COLOR,
        linewidth=LINEWIDTH_DIAG,
    )

    # Display Binned ECE
    ece_pct = bin_data["ece"] * 100
    ax_top.text(
        0.98, 0.02,
        f"Binned ECE={ece_pct:.2f}%",
        color="black",
        ha="right",
        va="bottom",
        transform=ax_top.transAxes,
        fontsize=FONT_SIZE_ECE,
    )

    ax_top.set_xlim(0, 1)
    ax_top.set_ylim(0, 1)
    ax_top.set_aspect("equal")
    ax_top.set_title(title, fontsize=FONT_SIZE_TITLE, fontweight="bold")
    ax_top.set_xlabel("")
    ax_top.set_ylabel("Expected Accuracy", fontsize=FONT_SIZE_LABEL)

    ax_top.legend(
        handles=[gap_bars, acc_bars],
        labels=["Gap", "Accuracy"],
        loc="upper right",
        fontsize=FONT_SIZE_LEGEND,
        framealpha=0.8,
    )

    # --- BOTTOM: Confidence Histogram (inverted) ---
    ax_bot.bar(
        positions,
        -counts,
        width=bin_size * HIST_WIDTH_FACTOR,
        color="steelblue",
        alpha=0.6,
        edgecolor="steelblue",
    )

    ax_bot.set_xlim(0, 1)
    ax_bot.set_xlabel("Confidence", fontsize=FONT_SIZE_LABEL)
    ax_bot.set_ylabel("Count", fontsize=FONT_SIZE_LABEL)

    yticks = ax_bot.get_yticks()
    ax_bot.set_yticklabels([str(int(abs(t))) for t in yticks])


# --- PLOT ---
title = (
    f"{TARGET_MODEL}\n"
    f"Stratified: {TARGET_STRATIFIED} | Calibrated: {TARGET_CALIBRATED}"
)

fig, (ax_top, ax_bot) = plt.subplots(
    nrows=2,
    ncols=1,
    sharex=True,
    figsize=FIGSIZE,
    dpi=DPI,
    gridspec_kw={"height_ratios": [HEIGHT_RATIO_DIAG, HEIGHT_RATIO_HIST]},
)

plot_reliability_binned(
    ax_top=ax_top,
    ax_bot=ax_bot,
    bin_data=bin_data,
    title=title,
)

plt.tight_layout()
plt.subplots_adjust(hspace=HSPACE_COMBINED)

# --- SAVE AS SVG ---
os.makedirs(OUTPUT_DIR, exist_ok=True)
seed_part = "mean" if USE_MEAN else f"seed_{TARGET_SEED}"
filename = (
    f"reliability_binned_{TARGET_MODEL}_{seed_part}"
    f"_strat_{str(TARGET_STRATIFIED).lower()}"
    f"_calib_{str(TARGET_CALIBRATED).lower()}.svg"
)

plt.savefig(
    os.path.join(OUTPUT_DIR, filename),
    format="svg",
    dpi=DPI,
    bbox_inches="tight",
)
print(f"✅ Saved: {filename}")
plt.show()   

In [ ]:
import pandas as pd

# Group by model, path, and true_label
df_model_summary = combined_df.groupby(['model', 'path', 'true_label'], as_index=False).agg(
    mean_pbb_score=('pbb_score', 'mean'),
    mean_pred_label=('predicted_label', 'mean') # Average of 0/1 predictions across seeds
)

# Binarize the averaged prediction (Majority Vote)
# If the model predicted '1' in >50% of seeds, the final prediction is 1.
df_model_summary['final_pred'] = (df_model_summary['mean_pred_label'] >= 0.5).astype(int)

print(df_model_summary.sample(5))

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
import seaborn as sns

models = df_model_summary['model'].unique()
n_models = len(models)

fig, axes = plt.subplots(1, n_models, figsize=(5 * n_models, 4.5))
if n_models == 1: axes = [axes]

for idx, model_name in enumerate(models):
    ax = axes[idx]
    
    # Filter the pre-aggregated dataframe for this specific model
    data = df_model_summary[df_model_summary['model'] == model_name]
    
    y_true = data['true_label']
    y_pred = data['final_pred']
    
    # Calculate Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    
    # Plot
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No-Stress', 'Stress'])
    disp.plot(ax=ax, cmap='Blues', values_format='d', colorbar=False)
    
    # Styling
    ax.set_title(f'{model_name}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    
    # Remove axis labels for inner plots if desired, or keep for clarity
    if idx != 0:
        ax.set_ylabel('')

# plt.suptitle('Confusion Matrices (Majority Vote per Model)', fontsize=14, y=1.02)
plt.tight_layout()
# plt.savefig('confusion_matrices_per_model.svg', format='svg', dpi=300, bbox_inches='tight')
plt.show()   

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix
import seaborn as sns

# 1. Prepare Data (Ensure df_model_summary exists from previous step)
models = df_model_summary['model'].unique()
n_models = len(models)

fig, axes = plt.subplots(1, n_models, figsize=(6 * n_models, 5))
if n_models == 1: axes = [axes]

# Create a global colorbar axis for the legend
cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.7]) # [left, bottom, width, height]

for idx, model_name in enumerate(models):
    ax = axes[idx]
    data = df_model_summary[df_model_summary['model'] == model_name]
    
    y_true = data['true_label']
    y_pred = data['final_pred']
    
    # Calculate Integer and Normalized Matrices
    cm_int = confusion_matrix(y_true, y_pred)
    cm_norm = cm_int.astype('float') / cm_int.sum(axis=1)[:, np.newaxis]
    
    # Plot Heatmap manually to control annotations
    # We plot the normalized values for color, but annotate with both
    sns.heatmap(cm_norm, annot=False, cmap='Blues', ax=ax, cbar=(idx==0), 
                cbar_ax=cbar_ax if idx == 0 else None, vmin=0, vmax=1)
    
    # Add Custom Annotations (Integer + Percentage)
    for i in range(cm_int.shape[0]):
        for j in range(cm_int.shape[1]):
            int_val = cm_int[i, j]
            norm_val = cm_norm[i, j] * 100
            # Format: "Count\n(Percentage%)"
            text_str = f"{int_val}\n({norm_val:.1f}%)"
            ax.text(j + 0.5, i + 0.5, text_str, ha='center', va='center', 
                    color='black' if norm_val < 50 else 'white', fontsize=10, fontweight='bold')

    # Axis Labels
    ax.set_title(f'{model_name}', fontsize=14, fontweight='bold', pad=15)
    ax.set_xlabel('Predicted', fontsize=12)
    ax.set_ylabel('True', fontsize=12)
    ax.set_xticks([0.5, 1.5])
    ax.set_yticks([0.5, 1.5])
    ax.set_xticklabels(['Negative', 'Positive'])
    ax.set_yticklabels(['Negative', 'Positive'])

# Add a global legend/title for the colorbar
cbar_ax.set_ylabel('Normalized Frequency', rotation=270, labelpad=20, fontsize=12)
cbar_ax.set_yticks([0, 0.5, 1])
cbar_ax.set_yticklabels(['0%', '50%', '100%'])

# plt.suptitle('Confusion Matrices: Counts and Normalized Percentages', fontsize=16, y=1.02)
plt.tight_layout(rect=[0, 0, 0.9, 1]) # Adjust right margin for colorbar
# plt.savefig('confusion_matrices_dual_annotation.svg', format='svg', dpi=300, bbox_inches='tight')
plt.show()